# Raw volume → Bronze

Reads the files each ingestion notebook landed for `run_date`, adds three audit
columns and writes a Delta table. **No business logic** — column names, types and
values are exactly as they arrived. Cleaning happens in Silver.

This is the one layer where a loop beats separate notebooks: the work is
genuinely identical for every source, so it is driven by the ingestion configs.

Idempotency: the write uses `replaceWhere _load_date = run_date`, so re-running a
day replaces exactly that day's rows and leaves every other day alone.

In [ ]:
import sys
from datetime import date
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from common_utils.ingestors import read_files
from common_utils.logger import get_logger, log_info
from common_utils.metadata import add_metadata_columns
from common_utils.observability import ensure_ops_schema, new_run_id, track
from common_utils.settings import load_json_folder, parse_run_date
from common_utils.writers import cluster_by, create_namespace, set_table_properties, write_idempotent

In [ ]:
dbutils.widgets.text("config_folder", "ingestion/config")
dbutils.widgets.text("sources", "")  # optional comma-separated subset, e.g. "s3_sales"
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("raw_volume", "raw_data")
dbutils.widgets.text("run_date", date.today().isoformat())

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
raw_volume = dbutils.widgets.get("raw_volume")
run_date = parse_run_date(dbutils.widgets.get("run_date"))
run_id = new_run_id()

configs = load_json_folder(dbutils.widgets.get("config_folder"))
selected = {s.strip() for s in dbutils.widgets.get("sources").split(",") if s.strip()}
sources = [c for name, c in configs.items() if not selected or name in selected]
logger = get_logger("bronze")

In [ ]:
create_namespace(spark, catalog, bronze_schema, raw_volume, comment="Bronze: raw copies of source data with audit columns")
ensure_ops_schema(spark, catalog)

failures = {}
for config in sources:
    source_name = config["source_name"]
    table = config["bronze_table"]
    path = f"/Volumes/{catalog}/{bronze_schema}/{raw_volume}/{source_name}/load_date={run_date}"
    try:
        with track(spark, catalog, run_id, run_date, task="raw_to_bronze", layer="bronze", entity=table) as stats:
            raw = read_files(spark, path, config["landing_format"])
            bronze = add_metadata_columns(raw, load_date=run_date)

            rows = bronze.count()
            write_idempotent(spark, bronze, catalog, bronze_schema, table, run_date)
            set_table_properties(spark, catalog, bronze_schema, table)
            cluster_by(spark, catalog, bronze_schema, table, ["_load_date"])

            stats.rows_read = stats.rows_written = rows
            log_info(logger, "bronze loaded", source=source_name, table=table, rows=rows, path=path)
    except Exception as exc:  # noqa: BLE001 - load every source, then report all failures together
        failures[table] = f"{type(exc).__name__}: {exc}"

if failures:
    raise RuntimeError(f"raw_to_bronze failed for {len(failures)} of {len(sources)} sources: {failures}")

In [ ]:
display(
    spark.sql(
        f"""
        SELECT '{run_date}' AS run_date, table_name, row_count FROM (
          SELECT 'customers' AS table_name, count(*) AS row_count FROM {catalog}.{bronze_schema}.customers WHERE _load_date = '{run_date}'
          UNION ALL SELECT 'sales_orders', count(*) FROM {catalog}.{bronze_schema}.sales_orders WHERE _load_date = '{run_date}'
          UNION ALL SELECT 'sales', count(*) FROM {catalog}.{bronze_schema}.sales WHERE _load_date = '{run_date}'
        )
        """
    )
)